In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [ ]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

In [5]:
from pyspark.sql import Function as F

# Top Travellers

## Problem Description

You are given two PySpark DataFrames: **`users`** and **`rides`**.

The `users` DataFrame contains information about users, while the `rides` DataFrame contains the distance travelled by each user.

Your task is to calculate the **total distance travelled by every user**.

Users who have **not taken any rides must still appear in the output**, with their travelled distance reported as `0`.

---

## Input DataFrames

### `users`

| Column | Data Type  | Description            |
| ------ | ---------- | ---------------------- |
| `id`   | int/string | Unique user identifier |
| `name` | string     | User name              |

### `rides`

| Column     | Data Type  | Description                      |
| ---------- | ---------- | -------------------------------- |
| `id`       | int/string | Unique ride identifier           |
| `user_id`  | int/string | ID of the user who took the ride |
| `distance` | int/string | Distance travelled in the ride   |

> Normalize `users.id` and `rides.user_id` to the same numeric type before joining. Cast `distance` to a numeric type before aggregation.

---

## Example Input

### Users

| id | name    |
| -: | ------- |
|  1 | Alice   |
|  2 | Bob     |
|  3 | Charlie |
|  4 | Diana   |

### Rides

| id | user_id | distance |
| -: | ------: | -------: |
|  1 |       1 |      120 |
|  2 |       1 |      100 |
|  3 |       2 |       50 |
|  4 |       2 |       80 |
|  5 |       1 |       60 |

---

## Expected Output

| name    | travelled_distance |
| ------- | -----------------: |
| Alice   |                280 |
| Bob     |                130 |
| Charlie |                  0 |
| Diana   |                  0 |

---

## Calculation

| User    | Ride Distances | Total |
| ------- | -------------- | ----: |
| Alice   | 120 + 100 + 60 |   280 |
| Bob     | 50 + 80        |   130 |
| Charlie | No rides       |     0 |
| Diana   | No rides       |     0 |

---

## Task

Return the following columns:

| Column               | Description                          |
| -------------------- | ------------------------------------ |
| `name`               | User name                            |
| `travelled_distance` | Total distance travelled by the user |

The result must be sorted by:

1. `travelled_distance` in **descending** order.
2. `name` in **ascending** order when two users have the same travelled distance.


## Problem Pattern

**Type Normalization → LEFT JOIN → GROUP BY → SUM → COALESCE → ORDER BY**


In [6]:
## PySpark Dataset


from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

users_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "Diana")
]

users = spark.createDataFrame(
    users_data,
    ["id", "name"]
)

rides_data = [
    (1, 1, 120),
    (2, 1, 100),
    (3, 2, 50),
    (4, 2, 80),
    (5, 1, 60)
]

rides = spark.createDataFrame(
    rides_data,
    ["id", "user_id", "distance"]
)

users.show()
rides.show()



+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
|  4|  Diana|
+---+-------+

+---+-------+--------+
| id|user_id|distance|
+---+-------+--------+
|  1|      1|     120|
|  2|      1|     100|
|  3|      2|      50|
|  4|      2|      80|
|  5|      1|      60|
+---+-------+--------+



# Using Spark SQL

In [7]:
users.createOrReplaceTempView("users")
rides.createOrReplaceTempView("rides")

In [15]:
spark.sql("""
    SELECT
        u.name,
        COALESCE(SUM(CAST(r.distance AS INT)), 0) AS travelled_distance
    FROM users u
    LEFT JOIN rides r
        ON CAST(u.id AS INT) = CAST(r.user_id AS INT)
    GROUP BY u.name
    ORDER BY travelled_distance DESC, u.name ASC
""").show()

+-------+------------------+
|   name|travelled_distance|
+-------+------------------+
|  Alice|               280|
|    Bob|               130|
|Charlie|                 0|
|  Diana|                 0|
+-------+------------------+



# Using Pyspark

In [20]:
from pyspark.sql.functions import *

In [31]:
results = users.join(
    rides,
    users.id.cast("int") == rides.user_id.cast("int"),
    "left"
).groupBy(
    users.name
).agg(
    sum(
        coalesce(rides.distance.cast("int"), lit(0))
    ).alias("travelled_distance")
).orderBy(
    col("travelled_distance").desc(),
    col("name").asc()
)

results.show()

+-------+------------------+
|   name|travelled_distance|
+-------+------------------+
|  Alice|               280|
|    Bob|               130|
|Charlie|                 0|
|  Diana|                 0|
+-------+------------------+

